#### HDBSCAN

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, davies_bouldin_score
from collections import Counter
import hdbscan


In [21]:
data = pd.read_csv("../../../data/preprocessed/preprocessed_dataset.csv")
y = data["expression"].astype("category").cat.codes

datasets = {
    "Scaled dataset": "../../../data/reduced/X_scaled.npy",
    "PCA dataset": "../../../data/reduced/X_pca.npy"
}

In [22]:
def remap_clusters_majority(y_true, y_pred):
    """
    Mapira svaki klaster na dominantnu pravu klasu,
    radi lakseg tumacenja i eventualnog confusion matrix prikaza.
    Noise (-1) ostaje -1.
    """
    mapped = np.full_like(y_pred, fill_value=-1)

    for cluster_id in np.unique(y_pred):
        if cluster_id == -1:
            continue

        mask = (y_pred == cluster_id)
        majority_class = Counter(y_true[mask]).most_common(1)[0][0]
        mapped[mask] = majority_class

    return mapped

In [23]:
def evaluate_hdbscan(y_true, labels):
    """
    Racuna metrike samo nad tackama koje nisu oznacene kao noise.
    """
    mask = labels != -1

    n_noise = np.sum(labels == -1)
    noise_ratio = n_noise / len(labels)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    if np.sum(mask) == 0 or n_clusters <= 1:
        return {
            "ari": -1,
            "nmi": -1,
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "noise_ratio": noise_ratio,
            "coverage": np.sum(mask) / len(labels)
        }

    ari = adjusted_rand_score(y_true[mask], labels[mask])
    nmi = normalized_mutual_info_score(y_true[mask], labels[mask])

    return {
        "ari": ari,
        "nmi": nmi,
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "noise_ratio": noise_ratio,
        "coverage": np.sum(mask) / len(labels)
    }


In [24]:
def plot_hdbscan_result(X, labels, dataset_name, title_suffix=""):
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        X[:, 0], X[:, 1],
        c=labels,
        s=8,
        cmap="tab10"
    )
    plt.title(f"HDBSCAN clustering - {dataset_name} {title_suffix}")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.colorbar(scatter, label="Cluster")
    plt.tight_layout()
    plt.show()


In [29]:
from joblib import Memory
memory = Memory("./hdbscan_cache", verbose=0)

min_cluster_sizes = [200, 500, 1000]
min_samples_values = [20, 50, 100]

results = []

for dataset_name, X_data in datasets.items():
    # if (dataset_name == "Scaled dataset"):
    #     continue

    # print("=" * 80)
    # print(dataset_name)
    # print("=" * 80)

    X = np.load(X_data)

    sample_size = min(5000, len(X))
    sample_idx = np.random.choice(len(X), size=sample_size, replace=False)

    X_tune = X[sample_idx]
    y_tune = y[sample_idx]

    best_labels = None
    best_ari = -1
    best_nmi = -1
    best_params = None

    for min_cluster_size in min_cluster_sizes:
        for min_samples in min_samples_values:

            model = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                cluster_selection_method="eom",
                memory=memory
            )

            labels = model.fit_predict(X_tune)

            mask = labels != -1
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            noise_points = np.sum(labels == -1)
            noise_ratio = noise_points / len(labels)
            if n_clusters < 2 or noise_ratio > 0.9 or np.sum(mask) < 2:
                continue

            ari = adjusted_rand_score(y_tune[mask], labels[mask])
            nmi = normalized_mutual_info_score(y_tune[mask], labels[mask])

            # print(
            #     f"min_cluster_size={min_cluster_size}, "
            #     f"min_samples={min_samples}, "
            #     f"ARI={ari:.4f}, "
            #     f"NMI={nmi:.4f}"
            # )

            if ari >= best_ari:
                best_ari = ari
                best_nmi = nmi
                best_labels = labels
                best_params = (min_cluster_size, min_samples)

    # print("\nBest params:", best_params)
    # print("Best ARI:", best_ari)
    # print("Best NMI:", best_nmi)

    # FINALNO fitovanje na celom dataset-u
    final_model = hdbscan.HDBSCAN(
        min_cluster_size=best_params[0],
        min_samples=best_params[1],
        cluster_selection_method="eom",
        memory=memory
    )

    final_labels = final_model.fit_predict(X)

    mask_full = final_labels != -1
    X_valid = X[mask_full]
    y_valid = y[mask_full]
    labels_valid = final_labels[mask_full]

    n_clusters_full = len(set(final_labels)) - (1 if -1 in final_labels else 0)
    noise_points_full = np.sum(final_labels == -1)
    noise_ratio_full = noise_points_full / len(final_labels)

    # ARI i NMI
    if np.sum(mask_full) >= 2 and n_clusters_full >= 2:
        final_ari = adjusted_rand_score(y_valid, labels_valid)
        final_nmi = normalized_mutual_info_score(y_valid, labels_valid)
    else:
        final_ari = -1
        final_nmi = -1

    # Silhouette i Davies-Bouldin
    # vaze samo ako ima bar 2 klastera i dovoljno tacaka
    if np.sum(mask_full) >= 2 and n_clusters_full >= 2 and len(np.unique(labels_valid)) >= 2:
        final_silhouette = silhouette_score(X_valid, labels_valid)
        final_db = davies_bouldin_score(X_valid, labels_valid)
    else:
        final_silhouette = -1
        final_db = -1

    results.append({
        "Dataset": dataset_name,
        "Algorithm": "HDBSCAN",
        "min_cluster_size": best_params[0],
        "min_samples": best_params[1],
        "Clusters": n_clusters_full,
        "Noise_points": noise_points_full,
        "Noise_ratio": round(noise_ratio_full, 4),
        "Silhouette": round(final_silhouette, 4) if final_silhouette != -1 else -1,
        "Davies_Bouldin": round(final_db, 4) if final_db != -1 else -1,
        "ARI": round(final_ari, 4),
        "NMI": round(final_nmi, 4)
    })

    plt.figure(figsize=(7, 6))
    plt.scatter(X[:, 0], X[:, 1], c=final_labels, s=8, cmap="tab10")
    plt.title(f"HDBSCAN - {dataset_name}")
    plt.tight_layout()
    plt.savefig(f"../../../results/hdbscan/hdbscan_{dataset_name.replace(' ', '_').lower()}.png")
    plt.close()
    # plt.show()

results_df = pd.DataFrame(results)
results_df.to_csv("../../../results/hdbscan/hdbscan_results_summary.csv", index=False)
display(results_df)

,Dataset,Algorithm,min_cluster_size,min_samples,Clusters,Noise_points,Noise_ratio,Silhouette,Davies_Bouldin,ARI,NMI
0,Scaled dataset,HDBSCAN,500,100,2,11554,0.4136,0.6497,0.5101,0.0068,0.0364
1,PCA dataset,HDBSCAN,1000,20,2,8206,0.2937,0.6220,0.5730,0.0055,0.0203


In [30]:
plt.figure(figsize=(7,6))
plt.scatter(X[:,0], X[:,1], c=y, s=8, cmap="tab10")
plt.title("True classes (expression)")
plt.tight_layout()
plt.savefig(f"../../../results/hdbscan/hdbscan_true_classes.png")
plt.close()
# plt.show()

In [31]:
classes, counts = np.unique(y, return_counts=True)

plt.figure(figsize=(7,5))
bars = plt.bar(classes, counts)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height,
             int(height),
             ha='center',
             va='bottom')

plt.xlabel("Expression class")
plt.ylabel("Number of samples")
plt.title("Distribution of expression classes")
plt.xticks(classes)
plt.tight_layout()
plt.savefig(f"../../../results/hdbscan/hdbscan_class_distribution.png")
plt.close()
# plt.show()

In [32]:
import seaborn as sns
import pandas as pd

df_plot = pd.DataFrame({
    "PC1": X[:,0],
    "PC2": X[:,1],
    "class": y
})

plt.figure(figsize=(7,6))

sns.kdeplot(
    data=df_plot,
    x="PC1",
    y="PC2",
    hue="class",
    levels=2,
    linewidths=1,
    alpha=0.5
)

plt.title("Density overlap of expression classes")
plt.tight_layout()
plt.savefig(f"../../../results/hdbscan/hdbscan_density_overlap.png")
plt.close()
# plt.show()